# 🐍 Day 02: Python for Data Engineering (2-3 YOE Interview Prep)

> **Goal:** Transition into an internal Data Engineering project at Tech Mahindra / clear 2-3 YOE client screening rounds.  
> **Focus:** Core data structures (Lists, Dicts, Sets), practical interview gotchas, and real-world ETL data wrangling logic — **without** third-party libraries (no pandas/numpy needed).

---

### 📌 How This Notebook Is Structured
1. **Section 1: Essential Data Structures & Tools for DE**  
   *The bread-and-butter Python operations you actually use on the job: sorting with `lambda`, dictionary lookups, set reconciliation, and date parsing.*
2. **Section 2: The 6 Classic Interview Gotchas**  
   *The exact trick questions interviewers ask to test if you understand Python memory, references, and scoping.*
3. **Section 3: Building Logical Thinking (Practical DE Coding Challenges)**  
   *Write real data transformation logic: Group & Aggregate, Dimension Lookup/Enrichment, Order-Preserving Deduplication, Data Cleaning, and Top-1 Per Department (Python's version of `ROW_NUMBER()`).*

---


## 🛠️ Section 1: Core Data Structures & Operations for DE

In daily Data Engineering tasks (and 30-minute coding rounds), you rarely write complex algorithms from scratch. Instead, you constantly manipulate **lists of dictionaries**, parse **messy strings/dates**, and reconcile **datasets**.


### 1.1 Sorting Complex Data with `lambda`
**Scenario:** In ETL pipelines, records often need to be sorted before writing to storage or before deduplicating (e.g., sort employees by salary descending, or orders by date).

**The Task:**
Given a list of employee dictionaries, write a function `sort_employees_by_salary(employees, descending=True)` that returns a new list sorted by `salary`.


In [6]:
from typing import List, Dict, Any

def sort_employees_by_salary(employees: List[Dict[str, Any]], descending: bool = True) -> List[Dict[str, Any]]:

    return sorted(employees, key=lambda o : (o["salary"]), reverse=descending)
    
# =====================================================================
# Tests: Run this cell to verify your code!
# =====================================================================
sample_employees = [
    {"name": "Pavan", "dept": "Data", "salary": 65000},
    {"name": "Amit", "dept": "QA", "salary": 45000},
    {"name": "Sneha", "dept": "Data", "salary": 85000},
    {"name": "Rahul", "dept": "DevOps", "salary": 55000}
]

res = sort_employees_by_salary(sample_employees, descending=True)
assert res[0]["name"] == "Sneha", "Highest earner should be first"
assert res[-1]["name"] == "Amit", "Lowest earner should be last"
assert res[1]["salary"] == 65000

print("✅ 1.1 Sorting with lambda passed!")


✅ 1.1 Sorting with lambda passed!


<details>
<summary>💡 Click to see solution & explanation</summary>

```python
def sort_employees_by_salary(employees, descending=True):
    return sorted(employees, key=lambda emp: emp["salary"], reverse=descending)
```
**Key Takeaway:** `sorted(iterable, key=lambda x: x['field'], reverse=True)` creates a new sorted list without modifying the original. If you want to sort in-place, use `employees.sort(key=...)`.
</details>


### 1.2 Dictionaries: Safe Key Access & Default Values
**Scenario:** Raw JSON / API data frequently has missing fields. If you write `row['age']` and `'age'` is missing, your entire ETL pipeline crashes with a `KeyError`.

**The Task:**
Write a function `extract_user_status(user_record: dict) -> str`:
- If `"status"` exists and is not empty, return its value.
- If `"status"` is missing or `None`, return `"UNKNOWN"`.
- Never let this function raise a `KeyError`!


In [ ]:

def extract_user_status(user_record: dict) -> str:
    status = user_record.get("status")
    return status if status else "UNKNOWN"
     

# =====================================================================
# Tests: Run this cell to verify your code!
# =====================================================================
assert extract_user_status({"id": 1, "status": "ACTIVE"}) == "ACTIVE"
assert extract_user_status({"id": 2}) == "UNKNOWN", "Missing key should return UNKNOWN"
assert extract_user_status({"id": 3, "status": None}) == "UNKNOWN", "None value should fallback to UNKNOWN"

print("✅ 1.2 Safe dictionary lookups passed!")


✅ 1.2 Safe dictionary lookups passed!


<details>
<summary>💡 Click to see solution & explanation</summary>

```python
def extract_user_status(user_record: dict) -> str:
    status = user_record.get("status")
    return status if status else "UNKNOWN"
```
**Key Takeaway:** `dict.get(key, default)` returns `default` (or `None`) if the key is missing instead of crashing with `KeyError`.
</details>


### 1.3 Sets: Data Reconciliation (Source vs Target Audit)
**Scenario:** In data warehousing, you often compare the primary keys in a source table vs a target table to detect missing records or orphan records.

**The Task:**
Write a function `reconcile_ids(source_ids: list, target_ids: list) -> dict` returning:
- `"missing_in_target"`: set of IDs present in source but missing in target.
- `"new_in_target"`: set of IDs present in target but not in source.
- `"matching_count"`: number of IDs present in both.


In [13]:
from typing import List, Dict, Set, Any

def reconcile_ids(source_ids: List[int], target_ids: List[int]) -> Dict[str, Any]:
    source_ids, target_ids = set(source_ids), set(target_ids)
    return {
        "missing_in_target": source_ids-target_ids,
        "new_in_target": target_ids-source_ids,
        "matching_count": len(source_ids & target_ids)
   }

# =====================================================================
# Tests: Run this cell to verify your code!
# =====================================================================
src = [101, 102, 103, 104, 105]
tgt = [103, 104, 105, 106, 107]

report = reconcile_ids(src, tgt)
assert report["missing_in_target"] == {101, 102}
assert report["new_in_target"] == {106, 107}
assert report["matching_count"] == 3  # 103, 104, 105

print("✅ 1.3 Set reconciliation passed!")


✅ 1.3 Set reconciliation passed!


<details>
<summary>💡 Click to see solution & explanation</summary>

```python
def reconcile_ids(source_ids, target_ids):
    src_set = set(source_ids)
    tgt_set = set(target_ids)
    return {
        "missing_in_target": src_set - tgt_set,
        "new_in_target": tgt_set - src_set,
        "matching_count": len(src_set & tgt_set)
    }
```
**Key Takeaway:** Checking `x in list` is $O(N)$ (slow for large data). Checking `x in set` is $O(1)$ instant lookup. Set subtraction `set_a - set_b` is the standard Python equivalent of SQL `EXCEPT` or `LEFT JOIN WHERE right.key IS NULL`.
</details>


### 1.4 String Cleaning & Datetime Parsing
**Scenario:** Raw CSV files often arrive with messy dates in multiple formats like `"15/08/2023"` or `"2023-08-15"`, with extra spaces. Before loading to SQL or Parquet, you must standardize them to `"YYYY-MM-DD"`.

**The Task:**
Write `parse_date_to_standard(date_str: str) -> str`:
- Strip leading/trailing whitespace.
- Support two common input formats: `DD/MM/YYYY` (e.g. `"25/12/2023"`) and `YYYY-MM-DD` (e.g. `"2023-12-25"`).
- Return standardized date string in format `YYYY-MM-DD`.
- If invalid or empty, return `None`.


In [ ]:
from datetime import datetime
from typing import Optional

def parse_date_to_standard(date_str: Optional[str]) -> Optional[str]:
    """Standardize date strings to YYYY-MM-DD."""
    # TODO: Clean string and try datetime.strptime with known formats
    pass

# =====================================================================
# Tests: Run this cell to verify your code!
# =====================================================================
assert parse_date_to_standard("  25/12/2023  ") == "2023-12-25"
assert parse_date_to_standard("2023-12-25") == "2023-12-25"
assert parse_date_to_standard("invalid_date") is None
assert parse_date_to_standard(None) is None

print("✅ 1.4 Date parsing passed!")


<details>
<summary>💡 Click to see solution & explanation</summary>

```python
from datetime import datetime

def parse_date_to_standard(date_str):
    if not date_str or not isinstance(date_str, str):
        return None
    cleaned = date_str.strip()
    for fmt in ("%d/%m/%Y", "%Y-%m-%d"):
        try:
            dt = datetime.strptime(cleaned, fmt)
            return dt.strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None
```
**Key Takeaway:** `strptime` = String Parse to Datetime object. `strftime` = Format Datetime object to String.
</details>


### 1.5 Pairing Headers with Rows using `zip`
**Scenario:** When reading CSV lines or database cursor results without column names, you have a header tuple `("id", "name", "city")` and a row of values `(101, "Pavan", "Hyderabad")`.

**The Task:**
Write a function `rows_to_dicts(headers: list, data_rows: list) -> list[dict]`:
- Takes a list of header strings and a list of row tuples/lists.
- Returns a list of dictionaries mapping column name to value.


In [ ]:
from typing import List, Tuple, Dict, Any

def rows_to_dicts(headers: List[str], data_rows: List[Tuple]) -> List[Dict[str, Any]]:
    """Convert list of headers and data rows into list of dictionaries."""
    # TODO: Use zip() and dict() in a list comprehension
    pass

# =====================================================================
# Tests: Run this cell to verify your code!
# =====================================================================
cols = ["id", "name", "role"]
rows = [
    (1, "Pavan", "Data Engineer"),
    (2, "Kalyan", "Analyst")
]
result = rows_to_dicts(cols, rows)
assert result == [
    {"id": 1, "name": "Pavan", "role": "Data Engineer"},
    {"id": 2, "name": "Kalyan", "role": "Analyst"}
]

print("✅ 1.5 rows_to_dicts passed!")


<details>
<summary>💡 Click to see solution & explanation</summary>

```python
def rows_to_dicts(headers, data_rows):
    return [dict(zip(headers, row)) for row in data_rows]
```
**Key Takeaway:** `zip(keys, values)` pairs up items `[(k1, v1), (k2, v2)]`, and `dict(...)` instantly turns those pairs into a dictionary. This is how data frameworks convert raw database cursor tuples into records.
</details>


---
## ⚠️ Section 2: The 6 Classic Interview Gotchas (2-3 YOE Level)

In Tech Mahindra internal evaluations and service company client interviews, interviewers rarely ask about obscure internals. Instead, they give you short 3-line snippets and ask:
> *"What does this print, and why?"*

Work through these 6 classic gotchas. Run each cell, observe the output, and master the explanation!


### ⚠️ Gotcha 2.1: Aliasing (`b = a`) vs Shallow Copy (`b = a.copy()`)
**The Code Snippet:**
```python
list_a = [1, 2, [3, 4]]
list_b = list_a
list_c = list_a.copy()

list_b[0] = 99
list_c[2].append(500)
```
**Question:** What are the values of `list_a`, `list_b`, and `list_c` now?


In [ ]:
list_a = [1, 2, [3, 4]]
list_b = list_a          # Aliasing: list_b points to the EXACT SAME list as list_a
list_c = list_a.copy()   # Shallow copy: new outer list, but inner list is shared!

list_b[0] = 99
list_c[2].append(500)

print("list_a:", list_a)
print("list_b:", list_b)
print("list_c:", list_c)


#### 🎯 Interview Explanation:
- `list_b = list_a`: Does **not** copy the list. It simply creates a second variable name pointing to the same list in memory. Changing `list_b` changes `list_a`.
- `list_c = list_a.copy()` (or `list_a[:]`): Creates a new outer list. Modifying `list_c[0] = 99` would **not** affect `list_a`.
- **The Trap:** The nested list `[3, 4]` is still shared between `list_a` and `list_c`! To make a completely independent copy of nested lists/dicts, you must use `copy.deepcopy()`.


### ⚠️ Gotcha 2.2: The Mutable Default Argument Bug
**The Code Snippet:**
```python
def add_event(event_id, event_list=[]):
    event_list.append(event_id)
    return event_list
```
**Question:** If I call `add_event("login")` and then `add_event("logout")`, what will the second call return?


In [ ]:
def add_event(event_id, event_list=[]):
    event_list.append(event_id)
    return event_list

print("Call 1:", add_event("login"))
print("Call 2:", add_event("logout"))
print("Default list ID in function:", id(add_event.__defaults__[0]))


#### 🎯 Interview Explanation:
- In Python, default arguments are **evaluated only once**, when the function is defined, **not** every time the function is called.
- The list `[]` was created once in memory. Every call without passing `event_list` reuses and modifies that same list!
- **How to fix it:** Always use `None` as the default value:
```python
def add_event(event_id, event_list=None):
    if event_list is None:
        event_list = []
    event_list.append(event_id)
    return event_list
```


### ⚠️ Gotcha 2.3: Modifying a List While Iterating Over It
**The Code Snippet:**
```python
numbers = [1, 2, 3, 4, 5, 6]
for n in numbers:
    if n % 2 == 0:
        numbers.remove(n)
```
**Question:** Will `numbers` contain only odd numbers `[1, 3, 5]` after this loop runs?


In [ ]:
numbers = [1, 2, 3, 4, 5, 6]
for n in numbers:
    if n % 2 == 0:
        numbers.remove(n)

print("Remaining numbers:", numbers)


#### 🎯 Interview Explanation:
- Notice that `4` was **not** removed!
- **Why?** When you remove `2` at index 1, all subsequent elements shift left. `3` moves to index 1, and `4` moves to index 2.
- The `for` loop internal pointer advances to index 2, which now contains `4`. So `3` was completely skipped!
- **How to fix it:** Never remove items from a list while iterating over it. Instead:
  1. Use a list comprehension: `numbers = [x for x in numbers if x % 2 != 0]`
  2. Or iterate over a slice copy: `for n in numbers[:]: if n % 2 == 0: numbers.remove(n)`


### ⚠️ Gotcha 2.4: `dict['key']` vs `dict.get('key')`
**The Question:**
When should you use `d['key']` and when should you use `d.get('key')` in a data pipeline?


In [ ]:
config = {"database": "postgres", "port": 5432}

# Scenario A: Key exists
print("Database:", config["database"])
print("Database (.get):", config.get("database"))

# Scenario B: Key does not exist
# config["password"] -> Crashes with KeyError!
print("Password with .get():", config.get("password", "default_secret"))


#### 🎯 Interview Explanation:
- Use `d[key]` when the key is **mandatory** (e.g., Primary Key `record['id']`). If it's missing, you *want* it to fail fast so you know data is corrupted.
- Use `d.get(key, default)` when the key is **optional** (e.g., `record.get('middle_name', '')`).


### ⚠️ Gotcha 2.5: Can You Change Values Inside a Tuple?
**The Code Snippet:**
```python
my_tuple = (10, "Pavan", [1, 2])
my_tuple[2].append(99)
```
**Question:** Tuples are immutable in Python. Will line 2 raise a `TypeError`?


In [ ]:
my_tuple = (10, "Pavan", [1, 2])

try:
    my_tuple[2].append(99)
    print("Success! my_tuple is now:", my_tuple)
except TypeError as e:
    print("Error:", e)


#### 🎯 Interview Explanation:
- A tuple's **references** are immutable: you cannot do `my_tuple[2] = [5, 6]` or change which objects the tuple points to.
- However, if an object *inside* the tuple is mutable (like a `list` or a `dict`), the contents of that inner object can be modified!
- This is a favorite trick question in 2-3 YOE interviews.


### ⚠️ Gotcha 2.6: `==` vs `is` (Especially with `None`)
**The Code Snippet:**
```python
val = None
# Option A:
if val == None: ...
# Option B:
if val is None: ...
```
**Question:** Why do senior interviewers insist on `if val is None:` instead of `if val == None:`?


In [ ]:
# '==' checks VALUE equality (can be customized by __eq__)
# 'is' checks MEMORY IDENTITY (is it the exact same object in RAM?)

class SneakyObject:
    def __eq__(self, other):
        return True # Always says it equals anything!

obj = SneakyObject()
print("obj == None:", obj == None)   # True (misleading!)
print("obj is None:", obj is None)   # False (correct!)


#### 🎯 Interview Explanation:
- `==` calls the `__eq__` method, which can be overridden or slow.
- `is` checks whether two variables point to the **exact same memory address** (`id(a) == id(b)`).
- In Python, `None` is a **singleton** (there is only one `None` object in the entire runtime).
- Always use `is None` or `is not None`.


---
## 🧠 Section 3: Building Logical Thinking (Real DE Coding Challenges)

These are the exact kinds of problems given in 30-minute technical rounds for 2-3 YOE Data Engineers.  
Write your solutions in the code cells and run the tests!


### 3.1 Group & Aggregate: Category Totals & Averages
**Scenario:** In SQL, you write `SELECT category, SUM(amount), AVG(amount), COUNT(*) FROM transactions GROUP BY category`. In Python, you often need to do this exact aggregation in memory.

**The Task:**
Implement `aggregate_transactions(transactions: list[dict]) -> dict`:
- Return a dictionary mapping each `category` to:
  - `"total_amount"`: sum of amounts
  - `"count"`: number of transactions
  - `"avg_amount"`: average amount rounded to 2 decimal places


In [ ]:
from typing import List, Dict, Any

def aggregate_transactions(transactions: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    """Group transactions by category and calculate total, count, and average."""
    # TODO: Build an accumulator dictionary, then compute averages
    pass

# =====================================================================
# Tests: Run this cell to verify your code!
# =====================================================================
tx_data = [
    {"category": "Electronics", "amount": 1200},
    {"category": "Groceries", "amount": 150},
    {"category": "Electronics", "amount": 800},
    {"category": "Groceries", "amount": 250},
    {"category": "Clothing", "amount": 300}
]

res = aggregate_transactions(tx_data)

assert res["Electronics"]["total_amount"] == 2000
assert res["Electronics"]["count"] == 2
assert res["Electronics"]["avg_amount"] == 1000.0

assert res["Groceries"]["total_amount"] == 400
assert res["Groceries"]["count"] == 2
assert res["Groceries"]["avg_amount"] == 200.0

assert res["Clothing"]["total_amount"] == 300
assert res["Clothing"]["count"] == 1

print("✅ 3.1 aggregate_transactions passed all tests!")


<details>
<summary>💡 Click to see solution & explanation</summary>

```python
def aggregate_transactions(transactions):
    grouped = {}
    for tx in transactions:
        cat = tx["category"]
        amt = tx["amount"]
        if cat not in grouped:
            grouped[cat] = {"total_amount": 0, "count": 0}
        grouped[cat]["total_amount"] += amt
        grouped[cat]["count"] += 1

    result = {}
    for cat, stats in grouped.items():
        result[cat] = {
            "total_amount": stats["total_amount"],
            "count": stats["count"],
            "avg_amount": round(stats["total_amount"] / stats["count"], 2)
        }
    return result
```
**Mental Model:**
1. Loop once to collect running totals and counts in a helper dict.
2. Loop once over the categories to compute the final averages.
</details>


### 3.2 In-Memory Join / Dimension Lookup (Enrichment)
**Scenario:** You have a stream of employee fact records with `dept_id`. You have a department dimension table. You need to enrich each employee record with `dept_name` (like a `LEFT JOIN`).

**The Task:**
Implement `enrich_employees(employees: list[dict], departments: list[dict]) -> list[dict]`:
- For each employee, look up their `dept_id` in `departments`.
- Add `"dept_name"` to the employee record.
- If `dept_id` is not found, set `"dept_name": "Unknown"`.
- Must be fast: Build a dictionary lookup table first so lookup is $O(1)$!


In [ ]:
from typing import List, Dict, Any

def enrich_employees(
    employees: List[Dict[str, Any]], 
    departments: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:
    """Enrich employee records with department name using a hash map lookup."""
    # TODO: Step 1: Create a dict mapping dept_id -> dept_name
    # TODO: Step 2: Loop through employees and attach dept_name
    pass

# =====================================================================
# Tests: Run this cell to verify your code!
# =====================================================================
emp_list = [
    {"emp_id": 1, "name": "Pavan", "dept_id": 10},
    {"emp_id": 2, "name": "Rohan", "dept_id": 20},
    {"emp_id": 3, "name": "Meera", "dept_id": 99}  # Not in departments table!
]

dept_list = [
    {"dept_id": 10, "dept_name": "Data Engineering"},
    {"dept_id": 20, "dept_name": "Cloud Operations"}
]

enriched = enrich_employees(emp_list, dept_list)

assert enriched[0]["dept_name"] == "Data Engineering"
assert enriched[1]["dept_name"] == "Cloud Operations"
assert enriched[2]["dept_name"] == "Unknown", "Missing dept_id should default to Unknown"

print("✅ 3.2 enrich_employees passed all tests!")


<details>
<summary>💡 Click to see solution & explanation</summary>

```python
def enrich_employees(employees, departments):
    # Step 1: Create O(1) lookup dictionary
    dept_map = {d["dept_id"]: d["dept_name"] for d in departments}
    
    # Step 2: Enrich each employee
    result = []
    for emp in employees:
        row = dict(emp) # Make a copy to avoid mutating the original
        row["dept_name"] = dept_map.get(emp["dept_id"], "Unknown")
        result.append(row)
    return result
```
**Why interviewers love this:**  
Beginners use nested loops (`for emp in employees: for dept in departments: ...`), which is $O(N \times M)$ (very slow). Building a dictionary lookup table makes the total time $O(N + M)$!
</details>


### 3.3 Deduplicate While Preserving Original Order
**Scenario:** You receive a feed of customer transaction IDs: `[101, 105, 101, 102, 105, 103]`. You need to remove duplicates, but **preserve the first-seen order**.

**The Catch:**
If you do `list(set(ids))`, sets do NOT guarantee preserving original order!

**The Task:**
Implement `deduplicate_preserve_order(items: list) -> list`.


In [ ]:
from typing import List, Any

def deduplicate_preserve_order(items: List[Any]) -> List[Any]:
    """Deduplicate items while maintaining their original encounter order."""
    # TODO: Track seen items using a set, append new ones to a list
    pass

# =====================================================================
# Tests: Run this cell to verify your code!
# =====================================================================
input_ids = [101, 105, 101, 102, 105, 103, 101]
res = deduplicate_preserve_order(input_ids)

assert res == [101, 105, 102, 103], f"Expected [101, 105, 102, 103], got {res}"
assert deduplicate_preserve_order([]) == []
assert deduplicate_preserve_order(["A", "A", "A"]) == ["A"]

print("✅ 3.3 deduplicate_preserve_order passed all tests!")


<details>
<summary>💡 Click to see solution & explanation</summary>

```python
# Approach 1: Classic seen-set pattern (explicit and clear in interviews)
def deduplicate_preserve_order(items):
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result

# Approach 2: Python 3.7+ dict.fromkeys() one-liner
# def deduplicate_preserve_order(items):
#     return list(dict.fromkeys(items))
```
**Why this matters:** In Python 3.7+, standard dictionaries preserve insertion order. But demonstrating Approach 1 with `seen = set()` shows the interviewer that you understand $O(1)$ set lookup vs list scanning.
</details>


### 3.4 Messy Data Cleaning & Schema Validation
**Scenario:** In an ingestion step, you parse raw API payloads. Some rows have invalid prices, negative quantities, or `None`s. You need to validate and compute the clean record.

**The Task:**
Implement `clean_sales_records(raw_records: list[dict]) -> list[dict]`:
For each record:
1. `id` must be convertible to an integer.
2. `price` can be string with spaces (e.g. `"  49.99 "`): strip and convert to float. If invalid string like `"N/A"` or `None`, skip the record.
3. `quantity` must be convertible to int and must be **> 0**. If 0 or negative, skip the record.
4. If valid, compute `"total_amount": round(price * quantity, 2)` and append to results.


In [ ]:
from typing import List, Dict, Any

def clean_sales_records(raw_records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Validate and clean raw sales data."""
    # TODO: Loop through records with try-except to parse price and quantity safely
    pass

# =====================================================================
# Tests: Run this cell to verify your code!
# =====================================================================
dirty_data = [
    {"id": "1", "price": "  150.50 ", "quantity": "2"}, # Valid -> 301.0
    {"id": "2", "price": "invalid",   "quantity": "5"}, # Invalid price -> skip
    {"id": "3", "price": "20.00",     "quantity": "0"}, # Zero qty -> skip
    {"id": "4", "price": None,        "quantity": "1"}, # None price -> skip
    {"id": "5", "price": "10.00",     "quantity": "-2"},# Negative qty -> skip
    {"id": "6", "price": "25.00",     "quantity": "4"}  # Valid -> 100.0
]

cleaned = clean_sales_records(dirty_data)
assert len(cleaned) == 2, f"Expected 2 clean records, got {len(cleaned)}"
assert cleaned[0]["id"] == 1 and cleaned[0]["total_amount"] == 301.0
assert cleaned[1]["id"] == 6 and cleaned[1]["total_amount"] == 100.0

print("✅ 3.4 clean_sales_records passed all tests!")


<details>
<summary>💡 Click to see solution & explanation</summary>

```python
def clean_sales_records(raw_records):
    clean = []
    for r in raw_records:
        try:
            # Check price
            price_raw = r.get("price")
            if price_raw is None:
                continue
            price = float(str(price_raw).strip())
            
            # Check quantity
            qty = int(r.get("quantity", 0))
            if qty <= 0:
                continue
                
            record_id = int(r["id"])
            
            clean.append({
                "id": record_id,
                "price": price,
                "quantity": qty,
                "total_amount": round(price * qty, 2)
            })
        except (ValueError, TypeError):
            # Skip any malformed record
            continue
    return clean
```
</details>


### 3.5 Highest Earner Per Department (Python's `ROW_NUMBER()`)
**Scenario:** In SQL, you solved this on Day 1 using `ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) WHERE rnk = 1`. In Python interviews, interviewers love asking:
> *"How would you find the top earner in each department in pure Python?"*

**The Task:**
Implement `top_earner_per_dept(employees: list[dict]) -> dict[str, dict]`:
- Takes a list of employee dicts `{"name": str, "dept": str, "salary": int}`.
- Returns a dictionary mapping `dept` to the single employee dictionary who has the highest salary in that department.


In [ ]:
from typing import List, Dict, Any

def top_earner_per_dept(employees: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    """Return the highest-earning employee record for each department."""
    # TODO: Loop through employees and track the max salary employee per dept
    pass

# =====================================================================
# Tests: Run this cell to verify your code!
# =====================================================================
team = [
    {"name": "Pavan", "dept": "Data", "salary": 65000},
    {"name": "Sneha", "dept": "Data", "salary": 85000},
    {"name": "Amit", "dept": "QA", "salary": 45000},
    {"name": "Kiran", "dept": "QA", "salary": 52000},
    {"name": "Rahul", "dept": "DevOps", "salary": 60000}
]

top_earners = top_earner_per_dept(team)

assert top_earners["Data"]["name"] == "Sneha" and top_earners["Data"]["salary"] == 85000
assert top_earners["QA"]["name"] == "Kiran" and top_earners["QA"]["salary"] == 52000
assert top_earners["DevOps"]["name"] == "Rahul"

print("✅ 3.5 top_earner_per_dept passed all tests!")


<details>
<summary>💡 Click to see solution & explanation</summary>

```python
def top_earner_per_dept(employees):
    top_per_dept = {}
    for emp in employees:
        dept = emp["dept"]
        # If dept not seen yet, or this employee makes more than current dept leader:
        if dept not in top_per_dept or emp["salary"] > top_per_dept[dept]["salary"]:
            top_per_dept[dept] = emp
    return top_per_dept
```
**Why this is elegant:**  
Notice how clean this is: It runs in a single pass ($O(N)$ time complexity) without needing complex sorting or grouping libraries.
</details>


---
## 🎯 Summary Checklist for TechM DE Interviews

Before your interview or project evaluation, make sure you are comfortable with these patterns:
- [ ] Sorting lists of dicts using `key=lambda x: x['col']`
- [ ] Safe dictionary access with `.get(key, default)`
- [ ] Set reconciliation (`set_a - set_b` and `set_a & set_b`)
- [ ] Date conversion using `datetime.strptime` and `strftime`
- [ ] Why `list = []` as default argument is dangerous
- [ ] Why `list_b = list_a` does not create a new list
- [ ] Grouping & Aggregating records using a dictionary accumulator

You are ready! Work through these cells, write your own implementations, and run the assertions to verify your mastery!
